In [1]:
!git clone https://github.com/fburlacu/czsl-prj.git


Cloning into 'czsl-prj'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 102 (delta 39), reused 97 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 204.18 KiB | 1.59 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [2]:
%cd czsl-prj

/content/czsl-prj


In [1]:
!pip install ftfy regex tqdm scipy pandas
!pip install git+https://github.com/openai/CLIP.git

Defaulting to user installation because normal site-packages is not writeable
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-4l3js9g0
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-4l3js9g0
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369549 sha256=8b3d0f5a815a9cb65dd1d9271d45fe0a1b53230ce599e493a5f21bbd8c4890cb
  Stored in directory: /tmp/pip-e

In [2]:
!cat CSP/download_data.sh

# Copyright (c) Facebook, Inc. and its affiliates.
# All rights reserved.
#
# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.
#

CURRENT_DIR=$(pwd)

mkdir data
cd data

# download datasets and splits
wget -c http://wednesday.csail.mit.edu/joseph_result/state_and_transformation/release_dataset.zip -O mitstates.zip
wget -c http://vision.cs.utexas.edu/projects/finegrained/utzap50k/ut-zap50k-images.zip -O utzap.zip
wget -c https://senthilpurushwalkam.com/publications/compositional/compositional_split_natural.tar.gz -O compositional_split_natural.tar.gz
wget -c https://huggingface.co/datasets/nihalnayak/cgqa/resolve/main/cgqa.zip -O cgqa.zip


# MIT-States
unzip mitstates.zip 'release_dataset/images/*' -d mit-states/
mv mit-states/release_dataset/images mit-states/images/
rm -r mit-states/release_dataset
rename "s/ /_/g" mit-states/images/*

# UT-Zappos50k
unzip utzap.zip -d ut-zap50k/
mv ut-zap50k/ut-zap50k-images ut-zap

In [ ]:
!sh CSP/download_data.sh

In [1]:
%cd data
!wget https://nlp.stanford.edu/data/glove.6B.zip


/home/jovyan/FoMo/czsl-prj/data
--2026-05-17 13:54:57--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 

/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-05-17 13:54:58--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip.1’

glove.6B.zip.1       52%[=========>          ] 434.30M  5.14MB/s    eta 75s    ^C


In [12]:
import os

print(os.getcwd())

images_dir = "../data/mit-states/images/"

for folder in os.listdir(images_dir):
    if " " in folder:
        old_path = os.path.join(images_dir, folder)
        new_path = os.path.join(images_dir, folder.replace(" ", "_"))
        os.rename(old_path, new_path)




/home/jovyan/FoMo/czsl-prj/CSP


In [41]:
print(os.getcwd())

/home/jovyan/FoMo/czsl-prj/CSP


In [18]:
%cd CSP
!PYTHONPATH=/home/jovyan/FoMo/czsl-prj/GDE python -u train.py \
  --dataset mit-states \
  --clip_model ViT-L/14 \
  --experiment_name csp \
  --seed 0 \
  --epochs 20 \
  --lr 5e-05 \
  --attr_dropout 0.3 \
  --weight_decay 0.00001 \
  --train_batch_size 64 \
  --gradient_accumulation_steps 2 \
  --context_length 8 \
  --save_path dicts/mit-states/csp_model \
  --save_every_n 1

[Errno 2] No such file or directory: 'CSP'
/home/jovyan/FoMo/czsl-prj/CSP
training details
Namespace(experiment_name='csp', dataset='mit-states', lr=5e-05, weight_decay=1e-05, clip_model='ViT-L/14', epochs=20, train_batch_size=64, eval_batch_size=1024, evaluate_only=False, context_length=8, attr_dropout=0.3, save_path='dicts/mit-states/csp_model', save_every_n=1, save_model=False, seed=0, gradient_accumulation_steps=2)
# train pairs: 1262 | # val pairs: 600 | # test pairs: 800
# train images: 30338 | # val images: 10420 | # test images: 12995
100%|████████████████████████████████████████| 890M/890M [00:04<00:00, 232MiB/s]
model dtype torch.float16
soft embedding dtype torch.float32
epoch   1:   0%|                                        | 0/475 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/home/jovyan/FoMo/czsl-prj/CSP/train.py", line 218, in <module>
    model, optimizer = train_model(
  File "/home/jovyan/FoMo/czsl-prj/CSP/train.py", line 64, in train_model
    batch_fe